<a href="https://colab.research.google.com/github/RajeshworM/IMPDS_Datafrom_Portal/blob/main/Spike_datatransform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ============================================================
# HOW TO USE THIS IN GOOGLE COLAB
# ------------------------------------------------------------
# 1. Open a new Google Colab notebook (colab.research.google.com)
# 2. Copy each "# CELL n" block below into its own separate cell,
#    in order (Cell 1 first, then Cell 2, etc.)
# 3. Run each cell one at a time (Shift+Enter), reading the printed
#    output before moving to the next cell - several cells print
#    diagnostic checks you should look at before continuing.
# 4. When Cell 1 runs, it will prompt you to upload two files:
#       - your MASTER price panel (with msp already filled in,
#         production column blank)
#       - your PRODUCTION file (state, commodity, crop_season,
#         agri_year, production, ...)
#    Both .csv and .xlsx are supported.
# ============================================================


# ============================================================
# CELL 1: Import libraries and upload both files
# ============================================================
import pandas as pd
import numpy as np
from google.colab import files

print("Please upload your MASTER PRICE FILE (msp already filled, production blank)")
uploaded_master = files.upload()
master_filename = list(uploaded_master.keys())[0]

print("\nPlease upload your PRODUCTION FILE")
uploaded_prod = files.upload()
prod_filename = list(uploaded_prod.keys())[0]

def read_any(filename):
    if filename.lower().endswith('.csv'):
        return pd.read_csv(filename)
    else:
        return pd.read_excel(filename)

master = read_any(master_filename)
prod = read_any(prod_filename)

print("\nMaster file shape:", master.shape)
print("Production file shape:", prod.shape)
print("\nMaster columns:", list(master.columns))
print("Production columns:", list(prod.columns))


# ============================================================
# CELL 2: Clean text columns and parse the master date column
# ============================================================
def clean_text(x):
    if pd.isna(x):
        return x
    return str(x).strip()

master['state'] = master['state'].apply(clean_text)
master['commodity'] = master['commodity'].apply(clean_text)

prod['state'] = prod['state'].apply(clean_text)
prod['commodity'] = prod['commodity'].apply(clean_text)
prod['crop_season'] = prod['crop_season'].apply(clean_text)

# Master date is assumed to be DD-MM-YYYY (e.g. 01-01-2000).
# If your date column is already a proper date, this still works fine.
master['date'] = pd.to_datetime(master['date'], dayfirst=True)

print("Unique states in MASTER:")
print(sorted(master['state'].unique()))
print("\nUnique states in PRODUCTION:")
print(sorted(prod['state'].unique()))

print("\nUnique commodities in MASTER:")
print(sorted(master['commodity'].unique()))
print("\nUnique commodities in PRODUCTION:")
print(sorted(prod['commodity'].unique()))

print("\nUnique crop_season labels in PRODUCTION:")
print(sorted(prod['crop_season'].unique()))


# ============================================================
# CELL 3: Fix name mismatches - EDIT these dictionaries if needed
# ============================================================
# If CELL 2's printed lists show a state or commodity spelled
# differently between the two files, add a mapping here:
#   'name_used_in_production_file': 'name_used_in_master_file'

STATE_CROSSWALK = {
    # 'Orissa': 'Odisha',
    # 'Pondicherry': 'Puducherry',
}

COMMODITY_CROSSWALK = {
    # 'Paddy': 'Rice',
    # 'Arhar/Tur': 'Arhar',
}

prod['state'] = prod['state'].replace(STATE_CROSSWALK)
prod['commodity'] = prod['commodity'].replace(COMMODITY_CROSSWALK)

master_states = set(master['state'].unique())
master_commodities = set(master['commodity'].unique())
prod_states = set(prod['state'].unique())
prod_commodities = set(prod['commodity'].unique())

print("States in PRODUCTION but NOT in MASTER (these rows will be dropped):")
print(prod_states - master_states)

print("\nStates in MASTER but NOT in PRODUCTION (these will stay NA - check if this is expected):")
print(master_states - prod_states)

print("\nCommodities in PRODUCTION but NOT in MASTER (these rows will be dropped):")
print(prod_commodities - master_commodities)

print("\nCommodities in MASTER but NOT in PRODUCTION (these will stay NA - check if this is expected):")
print(master_commodities - prod_commodities)

# Keep only production rows that match a state & commodity in your master file
prod_clean = prod[
    prod['state'].isin(master_states) & prod['commodity'].isin(master_commodities)
].copy()

print(f"\nProduction rows kept after filtering: {len(prod_clean)} out of {len(prod)}")
print("If this dropped far more rows than expected, check the crosswalk lists above.")


# ============================================================
# CELL 4: Expand each seasonal production row into its monthly window
# ============================================================
# Marketing-window convention used (as discussed):
#   Kharif -> October (year1) through March (year1+1)
#   Rabi   -> April (year1+1) through September (year1+1)
# where agri_year is written like "2000-01" and year1 = 2000.
#
# If CELL 2 showed crop_season labels other than 'Kharif'/'Rabi'
# (e.g. 'Whole Year', 'Summer'), add them below with their own
# start month and number of months before re-running this cell.

SEASON_WINDOWS = {
    'Kharif': {'start_month': 10, 'start_year_offset': 0, 'n_months': 6},
    'Rabi':   {'start_month': 4,  'start_year_offset': 1, 'n_months': 6},
}

def expand_row(row):
    season = row['crop_season']
    if season not in SEASON_WINDOWS:
        return None  # unhandled season - flagged by the warning below
    try:
        y1 = int(str(row['agri_year']).split('-')[0])
    except Exception:
        return None
    win = SEASON_WINDOWS[season]
    start = pd.Timestamp(year=y1 + win['start_year_offset'], month=win['start_month'], day=1)
    dates = pd.date_range(start=start, periods=win['n_months'], freq='MS')
    return pd.DataFrame({
        'state': row['state'],
        'commodity': row['commodity'],
        'date': dates,
        'production': row['production'],
    })

unknown_seasons = set(prod_clean['crop_season'].unique()) - set(SEASON_WINDOWS.keys())
if unknown_seasons:
    print("WARNING: the following crop_season labels are not mapped and will be SKIPPED:")
    print(unknown_seasons)
    print("Add them to SEASON_WINDOWS above (with the correct start month/length), then re-run from Cell 4.\n")
else:
    print("All crop_season labels are recognized (Kharif / Rabi). Proceeding.\n")

expanded_list = [expand_row(r) for _, r in prod_clean.iterrows()]
expanded_list = [e for e in expanded_list if e is not None]
expanded = pd.concat(expanded_list, ignore_index=True)

print("Expanded monthly production rows created:", len(expanded))

# Sanity check: no state-commodity-date combination should appear twice
# (that would mean two seasons' windows overlap, which should not happen
# with the Kharif Oct-Mar / Rabi Apr-Sep convention).
dupes = expanded[expanded.duplicated(subset=['state', 'commodity', 'date'], keep=False)]
if len(dupes) > 0:
    print("\nWARNING: overlapping season windows detected for these rows - check SEASON_WINDOWS:")
    print(dupes.sort_values(['state', 'commodity', 'date']).head(20))
else:
    print("No overlapping season windows found - good, each month has at most one production value.")


# ============================================================
# CELL 5: Merge the expanded production values into the master panel
# ============================================================
master_merged = master.merge(
    expanded.rename(columns={'production': 'production_new'}),
    on=['state', 'commodity', 'date'],
    how='left'
)

master_merged['production'] = master_merged['production_new']
master_merged.drop(columns=['production_new'], inplace=True)

filled = master_merged['production'].notna().sum()
missing = master_merged['production'].isna().sum()
print(f"Rows with production filled: {filled}")
print(f"Rows with production still NA: {missing}")
print("NA rows are expected for: off-season months of single-season crops,")
print("and for any state/commodity not found in your production file at all.")


# ============================================================
# CELL 6: Spot-check the pattern for a kharif-only, rabi-only, and
# both-season crop - confirm it matches what we worked through by hand
# ============================================================
def show_pattern(commodity_name, state_name, start='2000-07-01', end='2002-12-01'):
    sample = master_merged[
        (master_merged['commodity'] == commodity_name) &
        (master_merged['state'] == state_name) &
        (master_merged['date'] >= start) &
        (master_merged['date'] <= end)
    ][['date', 'msp', 'production']]
    print(f"\n--- {commodity_name} in {state_name} ---")
    print(sample.to_string(index=False))

# Edit the state/commodity names below to match your data if needed
show_pattern('Arhar', 'Andhra Pradesh & Telangana')   # expect: kharif-only pattern (Oct-Mar filled, Apr-Sep NA)
show_pattern('Wheat', 'Punjab')                        # expect: rabi-only pattern (Apr-Sep filled, Oct-Mar NA)
show_pattern('Rice', 'Punjab')                          # expect: both-season pattern (no NA, value switches Apr & Oct)


# ============================================================
# CELL 7: Save the completed file and download it
# ============================================================
output_filename = 'master_panel_with_production.csv'
master_merged.to_csv(output_filename, index=False)
files.download(output_filename)
print(f"Saved and downloading: {output_filename}")

Please upload your MASTER PRICE FILE (msp already filled, production blank)


Saving price_database_master.xlsx to price_database_master (2).xlsx

Please upload your PRODUCTION FILE


Saving production_data.xlsx to production_data (3).xlsx

Master file shape: (48000, 16)
Production file shape: (8326, 8)

Master columns: ['state_id', 'state', 'commodity', 'crop_id', 'group', 'group_id', 'date', 'year', 'month', 'price', 'rainfall', 'cpial', 'wpiall', 'fuel', 'msp', 'production']
Production columns: ['state', 'commodity', 'crop_season', 'agri_year', 'production', 'production_unit', 'Available Year', 'Available Month']
Unique states in MASTER:
['Andhra Pradesh & Telangana', 'Assam', 'Bihar', 'Gujarat', 'Haryana', 'Karnataka', 'Madhya Pradesh', 'Maharashtra', 'Orissa', 'Punjab', 'Rajasthan', 'Tamilnadu', 'Uttar Pradesh', 'West Bengal']

Unique states in PRODUCTION:
['Andhra Pradesh & Telangana', 'Assam', 'Bihar', 'Gujarat', 'Haryana', 'Karnataka', 'Madhya Pradesh', 'Maharashtra', 'Orissa', 'Punjab', 'Rajasthan', 'Tamilnadu', 'Uttar Pradesh', 'West Bengal']

Unique commodities in MASTER:
['Arhar', 'Bajra', 'Cotton', 'Gram', 'Groundnut', 'Jowar', 'Jute', 'Maize', 'Moong',

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved and downloading: master_panel_with_production.csv


In [7]:
# ============================================================
# HOW TO USE THIS IN GOOGLE COLAB
# ------------------------------------------------------------
# Only needs ONE file: your merged master panel (the file that
# already has msp and production columns, production still
# blank/NA for some state-commodity combinations).
#
# This finds every state-commodity pair where production is
# ENTIRELY missing (100% NA across all months) - these are the
# ones that never matched anything in your production source file,
# so you know exactly what to go collect.
# ============================================================


# ============================================================
# CELL 1: Upload the merged master file
# ============================================================
import pandas as pd
from google.colab import files

print("Upload your MERGED MASTER file (with production column)")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

master = pd.read_csv(filename) if filename.lower().endswith('.csv') else pd.read_excel(filename)

master['state'] = master['state'].astype(str).str.strip()
master['commodity'] = master['commodity'].astype(str).str.strip()

print("Master file loaded:", master.shape)


# ============================================================
# CELL 2: Check completeness of 'production' for every state-commodity pair
# ============================================================
completeness = (
    master.groupby(['state', 'commodity'])['production']
    .agg(
        total_rows='count',            # counts non-null only... so use size instead below
    )
)

# Recompute properly: total months vs how many are filled
completeness = (
    master.groupby(['state', 'commodity'])['production']
    .agg(total_months='size', filled_months=lambda x: x.notna().sum())
    .reset_index()
)
completeness['missing_months'] = completeness['total_months'] - completeness['filled_months']
completeness['pct_filled'] = (completeness['filled_months'] / completeness['total_months'] * 100).round(1)

# The ones you actually need to go collect: ZERO production data at all
fully_missing = completeness[completeness['filled_months'] == 0].sort_values(['commodity', 'state'])

# Also show partially-missing ones, just for visibility (may just be normal off-season NA)
partially_missing = completeness[
    (completeness['filled_months'] > 0) & (completeness['missing_months'] > 0)
].sort_values('pct_filled')

print(f"State-commodity pairs with COMPLETELY missing production (0% filled): {len(fully_missing)}")
print(f"State-commodity pairs with SOME missing production (likely normal off-season gaps): {len(partially_missing)}")


# ============================================================
# CELL 3: Show and save the list you need to act on
# ============================================================
print("\n=== STATES & CROPS WITH NO PRODUCTION DATA AT ALL - collect these ===")
print(fully_missing[['state', 'commodity']].to_string(index=False))

print("\n=== For reference: partially filled pairs (check if these are just normal off-season gaps) ===")
print(partially_missing[['state', 'commodity', 'pct_filled', 'missing_months']].to_string(index=False))

fully_missing[['state', 'commodity']].to_csv('states_crops_missing_production.csv', index=False)
completeness.to_csv('production_completeness_all_combinations.csv', index=False)

files.download('states_crops_missing_production.csv')
files.download('production_completeness_all_combinations.csv')

print("\nDownloaded:")
print(" - states_crops_missing_production.csv        -> ONLY the state-commodity pairs with zero data (your collection list)")
print(" - production_completeness_all_combinations.csv -> full completeness % for every state-commodity, for reference")

Upload your MERGED MASTER file (with production column)


Saving master_panel_with_production.csv to master_panel_with_production (3).csv
Master file loaded: (48000, 16)
State-commodity pairs with COMPLETELY missing production (0% filled): 0
State-commodity pairs with SOME missing production (likely normal off-season gaps): 160

=== STATES & CROPS WITH NO PRODUCTION DATA AT ALL - collect these ===
Empty DataFrame
Columns: [state, commodity]
Index: []

=== For reference: partially filled pairs (check if these are just normal off-season gaps) ===
                     state   commodity  pct_filled  missing_months
             Uttar Pradesh       Wheat        20.0             240
Andhra Pradesh & Telangana       Onion        48.0             156
                     Assam      Potato        48.0             156
                     Assam       Onion        48.0             156
                     Bihar      Potato        48.0             156
                     Bihar       Moong        48.0             156
                     Bihar       Onion

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Downloaded:
 - states_crops_missing_production.csv        -> ONLY the state-commodity pairs with zero data (your collection list)
 - production_completeness_all_combinations.csv -> full completeness % for every state-commodity, for reference
